# 실습 문제 모범답안 - 400. Deep Agent

`400_Deep_Agent` 노트북 실습 문제(나만의 데이터 분석 에이전트)에 대한 모범답안 예시입니다.

### 구현 요약

본문의 흐름을 도서 판매 데이터에 그대로 적용합니다.

1. 도서 판매 데이터 `data/book_sales.csv` 생성
2. 요약을 저장하는 `save_summary` 커스텀 도구
3. `create_deep_agent`로 에이전트 생성 후 장르별 매출 분석 요청
4. 같은 thread_id로 후속 질문 (멀티턴)

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

## 1. 백엔드 설정 및 데이터 준비

In [ ]:
import csv
from deepagents.backends import LocalShellBackend

WORK_DIR = os.getcwd()

import locale

# 서브프로세스 출력은 부모 프로세스(주피터 커널)의 기본 인코딩으로 디코딩되므로,
# 자식 python의 출력 인코딩을 부모와 일치시킵니다 (한글 Windows: cp949).
# ':replace' — 인코딩할 수 없는 문자는 오류 대신 대체 문자로 처리
SUBPROCESS_ENC = locale.getpreferredencoding(False)

backend = LocalShellBackend(
    root_dir=WORK_DIR,
    env={
        "PATH": os.environ.get("PATH", "/usr/bin:/bin"),  # execute 도구가 python 명령을 찾는 데 필요
        "PYTHONIOENCODING": f"{SUBPROCESS_ENC}:replace",
    },
)

# 도서 판매 데이터 생성
data = [
    ["Date", "Genre", "Units Sold", "Revenue"],
    ["2025-09-01", "소설", 12, 180000],
    ["2025-09-02", "에세이", 8, 96000],
    ["2025-09-03", "과학", 5, 90000],
    ["2025-09-04", "소설", 15, 225000],
    ["2025-09-05", "에세이", 6, 72000],
    ["2025-09-06", "과학", 9, 162000],
    ["2025-09-07", "소설", 10, 150000],
    ["2025-09-08", "에세이", 11, 132000],
]

data_dir = os.path.join(WORK_DIR, "data")
os.makedirs(data_dir, exist_ok=True)
csv_path = os.path.join(data_dir, "book_sales.csv")

with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:
    writer = csv.writer(f)
    writer.writerows(data)

print(f"데이터 저장 완료: {csv_path} ({len(data) - 1}행)")

## 2. 커스텀 도구: save_summary

본문의 `save_report`와 같은 구조입니다.

In [ ]:
from langchain.tools import tool

@tool
def save_summary(text: str) -> str:
    """분석 요약을 data/book_summary.txt 파일에 저장합니다.

    Args:
        text: 저장할 요약 텍스트
    """
    file_path = os.path.join(WORK_DIR, "data", "book_summary.txt")
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(text)
    return f"요약이 '{file_path}'에 저장되었습니다."

## 3. Deep Agent 생성 및 장르별 매출 분석 요청

본문과 같이 Windows 호환을 위해 execute 도구로 pandas를 사용하도록 지시하고,
경로의 백슬래시는 슬래시로 변환해 전달합니다.

In [ ]:
import uuid
from langgraph.checkpoint.memory import InMemorySaver
from deepagents import create_deep_agent

agent = create_deep_agent(
    model="openai:gpt-5-mini",
    tools=[save_summary],
    backend=backend,
    system_prompt=(
        "당신은 데이터 분석 전문가입니다. "
        "계획을 세운 뒤 코드를 실행해 데이터를 분석하고, 결과를 한국어로 정리하세요."
    ),
    checkpointer=InMemorySaver(),
)

thread_id = str(uuid.uuid4())
config = {"configurable": {"thread_id": thread_id}}

csv_path_for_agent = csv_path.replace("\\", "/")

input_message = {
    "role": "user",
    "content": (
        f"'{csv_path_for_agent}' 경로의 CSV 파일을 분석해주세요. "
        "반드시 execute 도구를 사용하여 Python(pandas)으로 데이터를 읽고, "
        "장르별 총 매출과 판매량을 분석하세요. "
        "Windows cmd 환경이므로: python3가 아닌 python 명령을 사용하고, "
        "heredoc(<<)이나 여러 줄 명령은 동작하지 않으니 "
        "세미콜론으로 이어진 한 줄짜리 python -c 명령만 사용하세요. "
        "read_file이나 ls 도구는 사용하지 마세요. "
        "분석이 끝나면 save_summary 도구로 한국어 요약을 저장해주세요."
    ),
}

for step in agent.stream(
    {"messages": [input_message]},
    config,
    stream_mode="updates",
):
    for _, update in step.items():
        if update and (messages := update.get("messages")) and isinstance(messages, list):
            for message in messages:
                message.pretty_print()

## 4. 멀티턴 후속 질문 + 결과 확인

같은 `thread_id`로 후속 질문을 던지면, 에이전트가 이전 분석 결과를 기억한 채 답변합니다.

In [ ]:
followup_message = {
    "role": "user",
    "content": "방금 분석에서 가장 매출이 높은 장르는 무엇이었나요? 한 문장으로 답해주세요.",
}

for step in agent.stream(
    {"messages": [followup_message]},
    config,  # 동일한 thread_id
    stream_mode="updates",
):
    for _, update in step.items():
        if update and (messages := update.get("messages")) and isinstance(messages, list):
            for message in messages:
                message.pretty_print()

In [ ]:
# 저장된 요약 파일 확인
summary_path = os.path.join(WORK_DIR, "data", "book_summary.txt")

if os.path.exists(summary_path):
    with open(summary_path, "r", encoding="utf-8") as f:
        print("=== 저장된 분석 요약 ===")
        print(f.read())
else:
    print(f"요약 파일이 아직 생성되지 않았습니다: {summary_path}")

## 확인 포인트

- 스트리밍 출력에서 내장 도구(`write_todos`, `execute`)와 커스텀 도구(`save_summary`)가 호출되는 순서를 관찰하세요.
- 후속 질문에서 에이전트가 CSV를 다시 읽지 않고 **이전 분석 맥락**으로 답하면 멀티턴(체크포인터)이 동작하는 것입니다.
- `data/book_summary.txt` 파일이 생성되었으면 커스텀 도구 연동까지 완료된 것입니다.